In [10]:
import pandas as pd

FILE = "flights_data.csv"
TARGET = "DEP_DEL15"  # DEP_DEL15, DEP_DELAY, ARR_DEL15 or ARR_DELAY
FLOWN_ONLY = False
USE_WEATHER = True
USE_PLANE = True

## Loading the dataset: Cols

In [11]:
TARGETS = ["DEP_DEL15", "DEP_DELAY", "ARR_DEL15", "ARR_DELAY"]
PLANE = ["MAKER", "MODEL", "SEATS", "PLANE_AGE"]
RAW = ["MONTH", "DAY_OF_WEEK", "CRS_DEP_TIME", "CRS_ARR_TIME", "DEP_TIME_BLK",
       "CRS_ELAPSED_TIME", "OP_UNIQUE_CARRIER", "ORIGIN", "DEST", "DISTANCE",
       "CANCELLED", "DIVERTED"] + TARGETS + PLANE

assert TARGET in TARGETS, f"TARGET must be one of {TARGETS}"

header = pd.read_csv(FILE, nrows=0).columns
WEATHER = [c for c in header if c.startswith(("origin_", "dest_"))]

df = pd.read_csv(FILE, usecols=[c for c in header if c in RAW or c in WEATHER])
n_read = len(df)
print(df.shape)

(3488373, 40)


## Row cleaning
#### Since our task is to predict the flight delay, we drop the diverted and canceled flights because our prediction does not include these flights IF SETTING FLOWN ONLY IS TRUE

In [12]:
df = df.dropna(subset=[TARGET])
if FLOWN_ONLY:
    df = df[(df["CANCELLED"] == 0) & (df["DIVERTED"] == 0)]
df = df.reset_index(drop=True)

if TARGET.endswith("DEL15"):
    df[TARGET] = df[TARGET].astype("int8")

print(f"read {n_read:,} rows -> kept {len(df):,}")

read 3,488,373 rows -> kept 3,414,526


## Scheduled Time Standardization
#### `CRS_DEP_TIME` and `CRS_ARR_TIME` are scheduled in HHMM, but we need to store it in hour and min. Tree models might need minute of the day.

In [13]:
for prefix, col in [("dep", "CRS_DEP_TIME"), ("arr", "CRS_ARR_TIME")]:
    hhmm = pd.to_numeric(df[col]).astype(int)
    df[f"{prefix}_hour"] = hhmm // 100 % 24
    df[f"{prefix}_minute_of_day"] = df[f"{prefix}_hour"] * 60 + hhmm % 100

df[["dep_hour", "arr_hour", "dep_minute_of_day", "arr_minute_of_day"]].describe().loc[["min", "max"]]

,dep_hour,arr_hour,dep_minute_of_day,arr_minute_of_day
min,0.0,0.0,1.0,0.0
max,23.0,23.0,1439.0,1439.0


## Plane info
#### the strings are padded with spaces, same maker would appear in different names: AIRBUS and AIRBUS INDUSTRIE.
#### 0 seats means unknown.

In [14]:
if "MAKER" in df:
    df["MAKER"] = df["MAKER"].str.strip().str.split().str[0]
if "MODEL" in df:
    model = df["MODEL"].str.strip()
    df["MODEL"] = model.mask(model == "")
if "SEATS" in df:
    df["SEATS"] = df["SEATS"].mask(df["SEATS"] == 0)

### Col Type and features

In [15]:
CALENDAR = ["DAY_OF_WEEK"]
SCHEDULE = ["dep_hour", "arr_hour", "DEP_TIME_BLK", "CRS_ELAPSED_TIME"]
ROUTE = ["OP_UNIQUE_CARRIER", "ORIGIN", "DEST", "DISTANCE"]

FEATURES = CALENDAR + SCHEDULE + ROUTE
if USE_WEATHER:
    FEATURES += WEATHER
if USE_PLANE:
    FEATURES += PLANE
FEATURES = [c for c in FEATURES if c in df.columns]

## Col Encoding

In [16]:
# String columns: encode these in every model
CATEGORICAL = [c for c in ["OP_UNIQUE_CARRIER", "ORIGIN", "DEST", "DEP_TIME_BLK", "MAKER", "MODEL"]
               if c in FEATURES]
for col in CATEGORICAL:
    df[col] = df[col].astype("category")

#One hot features
ONEHOT_FOR_LINEAR = ["DAY_OF_WEEK", "dep_hour", "arr_hour"]

#Cleanup features no longer needed
df = df.drop(columns=["CANCELLED", "DIVERTED", "CRS_DEP_TIME", "CRS_ARR_TIME"])

print(len(FEATURES), "features:", FEATURES)
print("categorical:", CATEGORICAL)

33 features: ['DAY_OF_WEEK', 'dep_hour', 'arr_hour', 'DEP_TIME_BLK', 'CRS_ELAPSED_TIME', 'OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'DISTANCE', 'origin_precipitation', 'origin_snowfall', 'origin_snow_depth', 'origin_temperature_2m', 'origin_wind_speed_10m', 'origin_wind_gusts_10m', 'origin_wind_direction_10m', 'origin_cloud_cover', 'origin_weather_code', 'origin_pressure_msl', 'dest_precipitation', 'dest_snowfall', 'dest_snow_depth', 'dest_temperature_2m', 'dest_wind_speed_10m', 'dest_wind_gusts_10m', 'dest_wind_direction_10m', 'dest_cloud_cover', 'dest_weather_code', 'dest_pressure_msl', 'MAKER', 'MODEL', 'SEATS', 'PLANE_AGE']
categorical: ['OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'DEP_TIME_BLK', 'MAKER', 'MODEL']


## Validation Step

In [17]:
SPLITS = [
    ([1],             2),   # train Jan      -> validate Feb
    ([1, 2],          3),   # train Jan-Feb  -> validate Mar
    ([1, 2, 3],       4),   # train Jan-Mar  -> validate Apr
    ([1, 2, 3, 4],    5),   # train Jan-Apr  -> validate May
    ([1, 2, 3, 4, 5], 6),   # train Jan-May  -> TEST on Jun
]
CV_SPLITS, TEST_SPLIT = SPLITS[:4], SPLITS[4]

def split(train_months, eval_month):
    return df[df["MONTH"].isin(train_months)], df[df["MONTH"] == eval_month]

for train_months, eval_month in SPLITS:
    train, valid = split(train_months, eval_month)
    print(f"train {train_months[0]}-{train_months[-1]}: {len(train):>9,} rows | month {eval_month}: {len(valid):>9,} rows")

train 1-1:   518,774 rows | month 2:   503,615 rows
train 1-2: 1,022,389 rows | month 3:   594,946 rows
train 1-3: 1,617,335 rows | month 4:   592,748 rows
train 1-4: 2,210,083 rows | month 5:   606,420 rows
train 1-5: 2,816,503 rows | month 6:   598,023 rows


## MODEL
### 1. Encoding differs in each model: one-hot, ordinal
### 2. Missing values differ in each model: Fill or leave as NaN
### 3. Scaling numerical values